In [ ]:
#imports
import numpy as np
from sklearn.metrics import classification_report 
from prettytable import PrettyTable

Note: you may need to restart the kernel to use updated packages.


# Classe Perceptron

In [ ]:
# Perceptron de Rosenblatt
class Perceptron:
    def __init__(self, bias=1, step_param=0, learning_rate=0.1,
                 interval=(-0.5, 0.5), max_epochs=1000):
        self.b = bias
        self.sp = step_param
        self.lr = learning_rate
        self.I = interval
        self.max_epochs = max_epochs
        self.w = None
        self.epochs = 0
        self.total_adjustments = 0

    def _activation(self, x):
        activation = np.sum(xi * w for xi, w in zip(x, self.w)) # soma cada elemento preditor com seu peso correspondente
        return int(activation >= self.sp) # avalia se valor de ativação é maior que o parâmetro de step

    def _learn(self, x, y, pred):
        e = y - pred
        dw = self.lr * e * np.array(x)
        self.w = self.w + dw
        self.total_adjustments += 1

    def fit(self, X, y):
        X_bias = [x + [self.b] for x in X] # adiciona o valor de viés ao conjunto de entrada

        # Inicializa os pesos apenas uma vez
        if self.w is None:
            self.w = np.random.uniform(self.I[0], self.I[1], len(X_bias[0])) #utiliza a distribuição normal para amostragem de valores aleatórios

        for _ in range(self.max_epochs):
            errors = 0
            for x, yi in zip(X_bias, y):
                pred = self._activation(x)
                if pred != yi:
                    self._learn(x, yi, pred)
                    errors += 1   
            self.epochs += 1
            if errors == 0:
                break

    def predict(self, X):
        X_bias = [x + [self.b] for x in X]
        return [int(self._activation(x)) for x in X_bias]

# Parte I - Resolvendo um Problema Linearmente Separável

In [ ]:
# === PARTE 1 (ENXUTA) ===
# Requisitos: degrau (θ=0), eta=0.1, w0~U(-0.5,0.5), até convergência, imprimir infos e plotar reta.
import numpy as np
import matplotlib.pyplot as plt

# 1) Carrega dataAll (binário de floats) e separa X, y
data = np.fromfile("data/dataAll.txt", dtype=float).reshape(-1, 3)
X = data[:, :2]
y = data[:, 2].astype(int)

# 2) Monta X com bias explícito (1.0)
X_aug = np.hstack([np.ones((X.shape[0], 1)), X])  # [1, x1, x2]

# 3) Inicializa pesos em U(-0.5, 0.5)
rng = np.random.default_rng(42)
w = rng.uniform(-0.5, 0.5, size=(3,))  # (w0, w1, w2)
w_init = w.copy()

# 4) Define função degrau (θ=0) e treinamento (Rosenblatt) até convergência
def step(u):  # θ = 0
    return 1 if u >= 0.0 else 0

eta = 0.1
epochs = 0
total_adjustments = 0
max_epochs = 10000  # segurança

while epochs < max_epochs:
    errors = 0
    for xi, yi in zip(X_aug, y):
        u = float(np.dot(w, xi))
        y_hat = step(u)
        e = yi - y_hat
        if e != 0:
            w = w + eta * e * xi
            total_adjustments += 1
            errors += 1
    epochs += 1
    if errors == 0:
        break

# 5) Impressões exigidas
print("Pesos iniciais:", w_init)
print("Total de ajustes:", total_adjustments)
print("Épocas até a convergência:", epochs)

# 6) Gráfico com pontos e reta separadora
cls0 = X[y == 0]
cls1 = X[y == 1]

plt.figure(figsize=(6,5))
plt.scatter(cls0[:,0], cls0[:,1], c='red', s=18, label='Classe 0')
plt.scatter(cls1[:,0], cls1[:,1], c='blue', s=18, label='Classe 1')

# Reta: w0 + w1*x1 + w2*x2 = 0  =>  x2 = -(w0 + w1*x1)/w2
w0, w1, w2 = w
x1_min, x1_max = X[:,0].min() - 0.5, X[:,0].max() + 0.5
xx = np.linspace(x1_min, x1_max, 200)
if abs(w2) > 1e-12:
    yy = -(w0 + w1*xx)/w2
    plt.plot(xx, yy, '--', linewidth=2, label='Fronteira aprendida')
else:
    xv = -w0 / w1
    plt.axvline(xv, linestyle='--', linewidth=2, label='Fronteira aprendida')

plt.title('Perceptron – Parte I (dataAll)')
plt.xlabel('x1'); plt.ylabel('x2')
plt.legend(); plt.grid(True)
plt.show()


# Parte II - Experimentação

In [ ]:
table = PrettyTable()
table.field_names = ['Taxa de Aprendizado', 'Intervalo de Pesos',
                     'Quantidade de Ajustes', 'Menor número de épocas para convergência']

# Carregamento do dataset
dataset = np.fromfile('data/data1.txt').reshape(-1, 3)
X = dataset[:, :-1].tolist()
y = dataset[:, -1].astype(int).tolist()

# Parâmetros de variação
learning_rates = [0.4, 0.1, 0.01]
weights_intervals = [(-0.5, 0.5), (-100, 100)]

# Repetições
repetition = 10

# Loop principal de teste
for lr in learning_rates:
    for interval in weights_intervals:

        errors = []
        epochs = []

        for i in range(repetition):
            p = Perceptron(learning_rate=lr, interval=interval)
            p.fit(X, y)

            errors.append(p.total_adjustments)
            epochs.append(p.epochs)
            
        mean = np.mean(errors)
        std = np.std(errors)

        table.add_row([
            f'η = {lr}',
            f'{interval}',
            f'{mean:.4f} ± {std:.4f}',
            np.min(epochs)
        ])

# Exibe a tabela
display(table)

C:\Users\Felix\AppData\Local\Temp\ipykernel_16304\272336435.py:15: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  activation = np.sum(xi * w for xi, w in zip(x, self.w)) # soma cada elemento preditor com seu peso correspondente


Taxa de Aprendizado,Intervalo de Pesos,Quantidade de Ajustes,Menor número de épocas para convergência
η = 0.4,"(-0.5, 0.5)",40.8000 ± 14.6410,8
η = 0.4,"(-100, 100)",393.6000 ± 193.2631,2
η = 0.1,"(-0.5, 0.5)",38.6000 ± 22.6592,3
η = 0.1,"(-100, 100)",1923.6000 ± 796.2500,16
η = 0.01,"(-0.5, 0.5)",83.0000 ± 42.7083,2
η = 0.01,"(-100, 100)",12495.4000 ± 5942.9013,96


# Discussão Crítica

## Taxa de Aprendizado

A taxa de aprendizado mostra uma relação significativa com o intervalo de inicialização dos pesos. Embora taxas maiores, como `η = 0.4`, geralmente levem a uma convergência mais rápida, isso depende fortemente do intervalo dos pesos iniciais. Por exemplo, a combinação `η = 0.4` com o intervalo `(-100, 100)` apresenta desempenho comparável ao de `η = 0.01` com o intervalo `(-0.5, 0.5)` em termos de menor número de épocas. No entanto, a quantidade de ajustes necessários varia drasticamente entre os cenários, indicando que uma taxa maior nem sempre significa maior eficiência no processo de aprendizado como um todo.

## Intervalo de Pesos

O intervalo de inicialização dos pesos tem um impacto direto na eficiência do treinamento. Utilizar um intervalo muito amplo, como `(-100, 100)`, tende a tornar o processo de convergência significativamente mais ineficiente — tanto em número de ajustes quanto em variação dos resultados. Em contraste, intervalos menores como `(-0.5, 0.5)` promovem uma convergência mais estável e com menos ajustes, mesmo quando utilizadas taxas de aprendizado pequenas. A quantidade de ajustes cresce de forma quase exponencial com o aumento do intervalo, sugerindo que inicializações com valores mais concentrados em torno de zero favorecem a aprendizagem do Perceptron.

## Conclusão

A análise evidencia que tanto a taxa de aprendizado quanto o intervalo de inicialização dos pesos têm um papel crucial na eficiência do Perceptron. Enquanto taxas mais altas podem acelerar a convergência, isso só ocorre de forma consistente quando os pesos são inicializados em intervalos moderados. Já intervalos muito amplos introduzem instabilidade e aumentam significativamente o número de ajustes, mesmo com taxas de aprendizado pequenas.

Portanto, para aplicações práticas, recomenda-se utilizar taxas de aprendizado moderadas (`η = 0.1` ou `η = 0.4`) combinadas com intervalos de pesos mais restritos (como `(-0.5, 0.5)`), garantindo melhor desempenho e menor custo computacional. Ajustes excessivos e lentidão no treinamento podem ser evitados com escolhas apropriadas desses hiperparâmetros.
